# 22. 플랫폼화 전후 효율성 변화 분석

## 분석 배경 및 목적

택시 호출 플랫폼(카카오T, 타다 등)은 **실시간 수요-공급 매칭 알고리즘**을 통해 전통적 택시 시장의 비효율성(빈차 배회, 정보 비대칭)을 근본적으로 변화시켰다. 이 변화가 운영 효율성과 소득 분배에 미친 영향을 실증적으로 측정하는 것은 플랫폼 규제 정책의 핵심 근거가 된다.

Cramer & Krueger (2016)은 NBER 워킹 페이퍼에서 미국 Uber의 효율성 효과를 정밀 측정하였다:
- **실차율(capacity utilization rate)**: UberX 운전자 50.2% vs 전통 택시 39.7% (LA), 41.2% vs 31.1% (NYC). 플랫폼 도입으로 실차율이 약 **30% 향상**되었다.
- 효율성 개선의 원천: (1) 실시간 GPS 기반 매칭으로 빈차 배회 시간 감소, (2) 수요 예측 기반 사전 배치, (3) 양면 시장(two-sided market) 구조의 네트워크 효과.

소득 불평등 측면에서, 플랫폼의 **알고리즘 배차**는 양면적 효과를 가진다:
- Gini 계수(Gini, 1912)는 소득/매출 분배의 불평등도를 0(완전 균등)-1(완전 불평등)로 측정하는 표준 지표이다.
- 플랫폼이 수요를 균등하게 분배하면 Gini가 감소하지만, 평점/이력 기반 우선 배차가 도입되면 오히려 불평등이 심화될 수 있다.

본 분석은 서울 택시 운행 데이터(2018-2025)를 활용하여 **실차율 추이**, **운전자 매출 불평등 지표(Gini, CV, P90/P10)**, **시간대별 빈차 패턴 변화**를 다각도로 측정한다.

| 데이터 | 용도 |
|--------|------|
| DC_TBYXD012 | 운행건별 승/하차시간, 거리, 매출, 운전자ID |
| calendar | 요일/공휴일 통제 |
| taxi_events | 플랫폼 이벤트 시점 참조 |

In [ ]:
# 필요 라이브러리 설치
# !pip install pandas numpy matplotlib seaborn psutil

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import platform
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# === 메모리 유틸 ===
import gc, psutil, os

def mem_usage(tag=''):
    gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'[MEM {tag}] {gb:.2f} GB')

CHUNK_SIZE = 1_000_000
mem_usage('start')

In [ ]:
# === 경로 설정 ===
D012_PATH = './DC_TBYXD012.csv'
EXT_DIR   = './external_data'
CALENDAR  = f'{EXT_DIR}/calendar_2018_2026.csv'
EVENTS    = f'{EXT_DIR}/taxi_events_timeline.csv'

## 0. 외부 데이터 로드

In [ ]:
cal = pd.read_csv(CALENDAR, encoding='utf-8', parse_dates=['date'])
events = pd.read_csv(EVENTS, encoding='utf-8', parse_dates=['date'])

# 플랫폼 관련 이벤트만
platform_events = events[events['category'] == 'platform']
print(f'캘린더: {len(cal):,}행 | 이벤트: {len(events):,}행')
platform_events[['date', 'event', 'description']]

## 1. 청크 집계: 연도별 실차율 및 운전자별 매출

**실차율(occupancy rate)** = 실차거리 / (실차거리 + 빈차거리)

Cramer & Krueger (2016)의 측정 방식을 따라, 거리 기반 실차율을 1차 지표로 사용한다. 시간 기반 실차율(실차시간 / 총 운행시간)은 보조 지표로 병행 산출한다.

- RIDE_DIST: 승차(실차) 거리
- VACNTV_DIST: 빈차 거리
- 실차시간 = ALIGHT_DTIME - RIDE_DTIME (이상치 3시간 초과 절삭)

In [ ]:
usecols = ['RIDE_DTIME', 'ALIGHT_DTIME', 'PAY_AMT', 'RIDE_DIST', 'VACNTV_DIST', 'DRIVER_ID']
dtypes = {
    'RIDE_DTIME': str, 'ALIGHT_DTIME': str,
    'PAY_AMT': 'float32', 'RIDE_DIST': 'float32',
    'VACNTV_DIST': 'float32', 'DRIVER_ID': str
}

# 누적 집계용
yearly_parts = []       # 연도-월별 실차율
driver_parts = []       # 운전자-연도별 매출
hour_year_parts = []    # 시간대-연도별 빈차시간

total_rows = 0
for i, chunk in enumerate(pd.read_csv(D012_PATH, usecols=usecols, dtype=dtypes, chunksize=CHUNK_SIZE)):
    rd = pd.to_datetime(chunk['RIDE_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    ad = pd.to_datetime(chunk['ALIGHT_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    m = rd.notna() & ad.notna()
    chunk = chunk[m].copy()
    rd, ad = rd[m], ad[m]
    
    chunk['year'] = rd.dt.year.astype('int16')
    chunk['month'] = rd.dt.month.astype('int8')
    chunk['hour'] = rd.dt.hour.astype('int8')
    chunk['date'] = rd.dt.normalize()
    
    # 실차시간(분)
    chunk['occ_min'] = (ad - rd).dt.total_seconds() / 60
    chunk['occ_min'] = chunk['occ_min'].clip(0, 180)  # 이상치 제거 (3시간 초과)
    
    # --- 1) 연도-월별 거리 기반 실차율 ---
    ym = chunk.groupby(['year', 'month']).agg(
        ride_dist_sum=('RIDE_DIST', 'sum'),
        vacntv_dist_sum=('VACNTV_DIST', 'sum'),
        occ_min_sum=('occ_min', 'sum'),
        trip_count=('PAY_AMT', 'count'),
        pay_sum=('PAY_AMT', 'sum')
    ).reset_index()
    yearly_parts.append(ym)
    
    # --- 2) 운전자-연도별 매출 ---
    drv = chunk.groupby(['DRIVER_ID', 'year']).agg(
        revenue=('PAY_AMT', 'sum'),
        trips=('PAY_AMT', 'count'),
        ride_dist=('RIDE_DIST', 'sum'),
        vacntv_dist=('VACNTV_DIST', 'sum')
    ).reset_index()
    driver_parts.append(drv)
    
    # --- 3) 시간대-연도-월별 빈차거리 ---
    hy = chunk.groupby(['year', 'month', 'hour']).agg(
        vacntv_sum=('VACNTV_DIST', 'sum'),
        ride_sum=('RIDE_DIST', 'sum'),
        cnt=('PAY_AMT', 'count')
    ).reset_index()
    hour_year_parts.append(hy)
    
    total_rows += len(chunk)
    if (i + 1) % 5 == 0:
        mem_usage(f'chunk {i+1}, rows={total_rows:,}')
    del chunk, rd, ad, ym, drv, hy
    gc.collect()

print(f'총 {total_rows:,}건 처리 완료')
mem_usage('after chunking')

## 2. 연도별 실차율 추이

카카오T가 2018년에 본격화된 이후, 실차율이 Cramer & Krueger (2016)가 보고한 미국 시장의 패턴(약 30% 향상)과 유사한 변화를 보이는지 확인한다. 단, 본 데이터는 2018년부터 시작하여 플랫폼 도입 이전의 baseline이 관측 불가하므로, 2018-2019년을 초기 플랫폼기로 설정하고 이후 추세를 관찰한다.

코로나 기간(2020-2022)의 외부 충격 효과를 분리하여 해석하는 것이 중요하다. 코로나로 인한 수요 감소가 실차율에 미치는 영향(수요 감소 -> 빈차 증가 -> 실차율 하락)은 플랫폼 효과와 반대 방향으로 작용한다.

In [ ]:
# 청크 합산
yearly = pd.concat(yearly_parts, ignore_index=True)
del yearly_parts; gc.collect()

yearly = yearly.groupby(['year', 'month']).sum().reset_index()

# 거리 기반 실차율 = 실차거리 / (실차거리 + 빈차거리)
yearly['occ_rate_dist'] = yearly['ride_dist_sum'] / (yearly['ride_dist_sum'] + yearly['vacntv_dist_sum'])

# 연도별 집계
yearly_agg = yearly.groupby('year').agg(
    ride_dist=('ride_dist_sum', 'sum'),
    vacntv_dist=('vacntv_dist_sum', 'sum'),
    occ_min=('occ_min_sum', 'sum'),
    trips=('trip_count', 'sum'),
    revenue=('pay_sum', 'sum')
).reset_index()

yearly_agg['occ_rate'] = yearly_agg['ride_dist'] / (yearly_agg['ride_dist'] + yearly_agg['vacntv_dist'])
yearly_agg['avg_occ_min'] = yearly_agg['occ_min'] / yearly_agg['trips']

print('=== 연도별 실차율 ===')
yearly_agg[['year', 'occ_rate', 'trips', 'avg_occ_min']].round(4)

In [ ]:
# 시각화: 연도별 실차율 추이
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 좌: 연도별 실차율
ax = axes[0]
ax.plot(yearly_agg['year'], yearly_agg['occ_rate'] * 100, 'o-', color='#1976d2', lw=2, ms=8)
ax.axvline(2018, color='red', ls='--', alpha=0.7, label='카카오T 본격화 (2018)')
ax.axvspan(2020, 2022, alpha=0.1, color='gray', label='코로나 기간')
ax.set_title('연도별 실차율 추이 (거리 기준)', fontsize=14)
ax.set_xlabel('연도'); ax.set_ylabel('실차율 (%)')
ax.legend()
ax.grid(True, alpha=0.3)

# 우: 월별 실차율 추이
ax = axes[1]
yearly['ym'] = yearly['year'].astype(str) + '-' + yearly['month'].astype(str).str.zfill(2)
yearly = yearly.sort_values('ym')
ax.plot(range(len(yearly)), yearly['occ_rate_dist'] * 100, color='#1976d2', lw=1)
ax.set_title('월별 실차율 추이', fontsize=14)
ax.set_xlabel('연-월'); ax.set_ylabel('실차율 (%)')
# x축 레이블 (연도 시작점만)
jan_idx = yearly[yearly['month'] == 1].index
ax.set_xticks([list(yearly.index).index(j) for j in jan_idx])
ax.set_xticklabels(yearly.loc[jan_idx, 'year'].values)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 운전자별 매출 불평등 분석 (Gini 계수, CV)

소득 불평등 측정에 사용되는 표준 지표들을 택시 운전자 매출 분포에 적용한다:

**Gini 계수 (Gini, 1912):**
- 0 = 완전 균등 (모든 운전자 동일 매출)
- 1 = 완전 불평등 (한 운전자가 모든 매출 독점)
- 로렌츠 곡선과 대각선 사이 면적의 2배로 정의됨

**변동계수 (CV):**
- 표준편차 / 평균. 평균 대비 상대적 분산을 측정
- 스케일에 독립적이므로 연도 간 비교에 적합

**P90/P10 비율:**
- 상위 10% 경계값 / 하위 10% 경계값
- 극단값에 덜 민감하면서 양 꼬리 격차를 직접 측정

플랫폼이 수요를 균등화시켰다면, 연도가 지남에 따라 세 지표 모두 감소하는 추세를 보여야 한다.

In [ ]:
# 운전자 데이터 합산
driver = pd.concat(driver_parts, ignore_index=True)
del driver_parts; gc.collect()

driver = driver.groupby(['DRIVER_ID', 'year']).agg(
    revenue=('revenue', 'sum'),
    trips=('trips', 'sum'),
    ride_dist=('ride_dist', 'sum'),
    vacntv_dist=('vacntv_dist', 'sum')
).reset_index()

# 최소 운행 건수 필터 (연간 100건 이상)
driver = driver[driver['trips'] >= 100].copy()
driver['occ_rate'] = driver['ride_dist'] / (driver['ride_dist'] + driver['vacntv_dist'])
driver['rev_per_trip'] = driver['revenue'] / driver['trips']

print(f'운전자-연도 조합: {len(driver):,}건')
driver.head()

In [ ]:
def gini_coefficient(values):
    """Gini 계수 계산 (0=완전균등, 1=완전불평등)"""
    v = np.sort(np.asarray(values, dtype=float))
    v = v[v > 0]
    n = len(v)
    if n == 0:
        return np.nan
    idx = np.arange(1, n + 1)
    return (2 * np.sum(idx * v) - (n + 1) * np.sum(v)) / (n * np.sum(v))

# 연도별 불평등 지표
inequality = []
for yr, grp in driver.groupby('year'):
    rev = grp['revenue'].values
    g = gini_coefficient(rev)
    cv = np.std(rev) / np.mean(rev) if np.mean(rev) > 0 else np.nan
    p90 = np.percentile(rev, 90)
    p10 = np.percentile(rev, 10)
    ratio_90_10 = p90 / p10 if p10 > 0 else np.nan
    p75 = np.percentile(rev, 75)
    p25 = np.percentile(rev, 25)
    
    inequality.append({
        'year': yr,
        'n_drivers': len(grp),
        'gini': g,
        'cv': cv,
        'p90_p10_ratio': ratio_90_10,
        'p75_p25_ratio': p75 / p25 if p25 > 0 else np.nan,
        'mean_revenue': np.mean(rev),
        'median_revenue': np.median(rev),
        'p10': p10,
        'p90': p90
    })

ineq_df = pd.DataFrame(inequality)
print('=== 연도별 운전자 매출 불평등 지표 ===')
ineq_df.round(4)

In [ ]:
# 시각화: Gini 계수 및 CV 추이
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (1) Gini 계수
ax = axes[0]
ax.plot(ineq_df['year'], ineq_df['gini'], 'o-', color='#d32f2f', lw=2, ms=8)
ax.axvline(2018, color='gray', ls='--', alpha=0.5, label='카카오T 본격화')
ax.set_title('연도별 Gini 계수 (운전자 매출)', fontsize=13)
ax.set_xlabel('연도'); ax.set_ylabel('Gini 계수')
ax.legend(); ax.grid(True, alpha=0.3)

# (2) 변동계수 (CV)
ax = axes[1]
ax.plot(ineq_df['year'], ineq_df['cv'], 's-', color='#388e3c', lw=2, ms=8)
ax.axvline(2018, color='gray', ls='--', alpha=0.5, label='카카오T 본격화')
ax.set_title('연도별 변동계수 (CV)', fontsize=13)
ax.set_xlabel('연도'); ax.set_ylabel('CV')
ax.legend(); ax.grid(True, alpha=0.3)

# (3) P90/P10 비율
ax = axes[2]
ax.plot(ineq_df['year'], ineq_df['p90_p10_ratio'], 'D-', color='#7b1fa2', lw=2, ms=8)
ax.axvline(2018, color='gray', ls='--', alpha=0.5, label='카카오T 본격화')
ax.set_title('연도별 P90/P10 매출 비율', fontsize=13)
ax.set_xlabel('연도'); ax.set_ylabel('P90/P10 비율')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. 운전자 매출 분포 변화 (연도별 비교)

In [ ]:
# 주요 연도별 매출 분포 비교 (바이올린 플롯)
years_to_compare = sorted(driver['year'].unique())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (1) 연도별 매출 박스 플롯
ax = axes[0]
data_for_box = [driver[driver['year'] == y]['revenue'].values for y in years_to_compare]
bp = ax.boxplot(data_for_box, labels=years_to_compare, patch_artist=True, showfliers=False)
colors_box = plt.cm.Blues(np.linspace(0.3, 1, len(years_to_compare)))
for patch, c in zip(bp['boxes'], colors_box):
    patch.set_facecolor(c)
ax.set_title('연도별 운전자 매출 분포', fontsize=13)
ax.set_xlabel('연도'); ax.set_ylabel('연간 총매출 (원)')
ax.grid(True, alpha=0.3)

# (2) 연도별 실차율 분포
ax = axes[1]
data_for_box2 = [driver[driver['year'] == y]['occ_rate'].dropna().values for y in years_to_compare]
bp2 = ax.boxplot(data_for_box2, labels=years_to_compare, patch_artist=True, showfliers=False)
colors_box2 = plt.cm.Greens(np.linspace(0.3, 1, len(years_to_compare)))
for patch, c in zip(bp2['boxes'], colors_box2):
    patch.set_facecolor(c)
ax.set_title('연도별 운전자 실차율 분포', fontsize=13)
ax.set_xlabel('연도'); ax.set_ylabel('실차율')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 상위/하위 운전자 간 격차 변화

Gini 계수가 전체 분포의 불평등도를 단일 숫자로 요약하는 반면, **상위/하위 분위 비교**는 분포의 꼬리(tail) 행태를 직접 관찰한다. 플랫폼의 알고리즘 배차가 하위 운전자의 매출을 끌어올리는 '하방 보호' 효과를 가지는지, 아니면 상위 운전자에게 수요를 집중시키는 '부익부' 효과를 가지는지 구분할 수 있다.

상위 10%와 하위 10% 운전자의 매출 추이를 비교하며, 격차 비율(top10_mean / bot10_mean)의 연도별 변화를 추적한다.

In [ ]:
# 연도별 상위/하위 10% 매출 및 실차율
tier_stats = []
for yr, grp in driver.groupby('year'):
    rev = grp['revenue']
    q10, q90 = rev.quantile(0.1), rev.quantile(0.9)
    
    top10 = grp[grp['revenue'] >= q90]
    bot10 = grp[grp['revenue'] <= q10]
    mid = grp[(grp['revenue'] > q10) & (grp['revenue'] < q90)]
    
    tier_stats.append({
        'year': yr,
        'top10_mean_rev': top10['revenue'].mean(),
        'bot10_mean_rev': bot10['revenue'].mean(),
        'mid_mean_rev': mid['revenue'].mean(),
        'top10_occ_rate': top10['occ_rate'].mean(),
        'bot10_occ_rate': bot10['occ_rate'].mean(),
        'top10_rev_per_trip': top10['rev_per_trip'].mean(),
        'bot10_rev_per_trip': bot10['rev_per_trip'].mean(),
        'gap_ratio': top10['revenue'].mean() / bot10['revenue'].mean() if bot10['revenue'].mean() > 0 else np.nan
    })

tier_df = pd.DataFrame(tier_stats)
print('=== 상위/하위 10% 매출 비교 ===')
tier_df.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (1) 상위/하위 매출 추이
ax = axes[0]
ax.plot(tier_df['year'], tier_df['top10_mean_rev'] / 1e6, 'o-', color='#d32f2f', label='상위 10%', lw=2)
ax.plot(tier_df['year'], tier_df['bot10_mean_rev'] / 1e6, 's-', color='#1976d2', label='하위 10%', lw=2)
ax.plot(tier_df['year'], tier_df['mid_mean_rev'] / 1e6, '^-', color='#388e3c', label='중간 80%', lw=2)
ax.axvline(2018, color='gray', ls='--', alpha=0.5)
ax.set_title('등급별 평균 매출 추이', fontsize=13)
ax.set_xlabel('연도'); ax.set_ylabel('평균 매출 (백만원)')
ax.legend(); ax.grid(True, alpha=0.3)

# (2) 격차 비율 (top10/bot10)
ax = axes[1]
ax.plot(tier_df['year'], tier_df['gap_ratio'], 'D-', color='#f57c00', lw=2, ms=8)
ax.axvline(2018, color='gray', ls='--', alpha=0.5, label='카카오T 본격화')
ax.set_title('상위 10% / 하위 10% 매출 격차 비율', fontsize=13)
ax.set_xlabel('연도'); ax.set_ylabel('격차 비율 (배)')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 시간대별 빈차시간 추이 (연도별 히트맵)

빈차 비율 = 빈차거리 / (실차거리 + 빈차거리). 이 비율이 높을수록 해당 시간대에 수요-공급 불일치가 크다는 의미이다.

시간대(0-23) x 연도 히트맵으로 빈차 패턴의 **시공간적 변화**를 시각화한다. 플랫폼의 매칭 알고리즘이 효과적이라면, 특히 수요가 불규칙한 심야/새벽 시간대에서 빈차율 감소가 두드러져야 한다.

코로나 전(2018-2019), 코로나 중(2020-2022), 코로나 후(2023-)의 3개 시기를 비교하여, 외부 충격과 플랫폼 효과를 구분한다.

In [ ]:
# 시간대-연도별 집계
hour_year = pd.concat(hour_year_parts, ignore_index=True)
del hour_year_parts; gc.collect()

hour_year = hour_year.groupby(['year', 'hour']).agg(
    vacntv_sum=('vacntv_sum', 'sum'),
    ride_sum=('ride_sum', 'sum'),
    cnt=('cnt', 'sum')
).reset_index()

# 빈차 비율 (높을수록 비효율)
hour_year['vacancy_rate'] = hour_year['vacntv_sum'] / (hour_year['vacntv_sum'] + hour_year['ride_sum'])

# 피벗: 시간대(행) x 연도(열)
pivot = hour_year.pivot_table(index='hour', columns='year', values='vacancy_rate')
pivot.round(3)

In [ ]:
# 히트맵 시각화
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    pivot * 100,
    cmap='RdYlGn_r',  # 빈차율 높을수록 빨간색
    annot=True, fmt='.1f',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': '빈차율 (%)'},
    ax=ax
)
ax.set_title('시간대 x 연도별 빈차율 (%) 히트맵', fontsize=14)
ax.set_xlabel('연도'); ax.set_ylabel('시간대')
plt.tight_layout()
plt.show()

In [ ]:
# 시간대별 빈차율 변화: 데이터 내 시기 비교
# 주의: D012 데이터는 2018-01부터 시작 → '플랫폼 이전(~2017)'은 데이터에 없음.
# 따라서 '플랫폼 전후'가 아니라 데이터 내 시기(코로나 전/후) 비교로 한정한다.
seg_pre_covid = hour_year[hour_year['year'].isin([2018, 2019])].groupby('hour').agg(
    vacntv=('vacntv_sum', 'sum'), ride=('ride_sum', 'sum')).reset_index()
seg_pre_covid['vacancy_rate'] = seg_pre_covid['vacntv'] / (seg_pre_covid['vacntv'] + seg_pre_covid['ride'])

seg_covid = hour_year[hour_year['year'].isin([2020, 2021, 2022])].groupby('hour').agg(
    vacntv=('vacntv_sum', 'sum'), ride=('ride_sum', 'sum')).reset_index()
seg_covid['vacancy_rate'] = seg_covid['vacntv'] / (seg_covid['vacntv'] + seg_covid['ride'])

seg_post_covid = hour_year[hour_year['year'] >= 2023].groupby('hour').agg(
    vacntv=('vacntv_sum', 'sum'), ride=('ride_sum', 'sum')).reset_index()
seg_post_covid['vacancy_rate'] = seg_post_covid['vacntv'] / (seg_post_covid['vacntv'] + seg_post_covid['ride'])

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(seg_pre_covid['hour'], seg_pre_covid['vacancy_rate'] * 100, 's-', label='코로나 전 (2018-2019)', lw=2)
ax.plot(seg_covid['hour'], seg_covid['vacancy_rate'] * 100, 'o-', label='코로나 (2020-2022)', lw=2)
ax.plot(seg_post_covid['hour'], seg_post_covid['vacancy_rate'] * 100, 'D-', label='코로나 후 (2023~)', lw=2)
ax.set_title('시간대별 빈차율 변화 (데이터 내 시기 비교)', fontsize=14)
ax.set_xlabel('시간대'); ax.set_ylabel('빈차율 (%)')
ax.set_xticks(range(24))
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('주의: 데이터가 2018부터라 카카오T 도입 이전 baseline은 관측 불가.')
print('따라서 이 비교는 플랫폼 인과효과가 아니라 시기별 기술 통계임.')

## 7. 캘린더 조인: 요일/공휴일별 실차율

요일과 공휴일 효과를 통제한 후에도 연도별 추세가 유지되는지 확인한다.

In [ ]:
# 월별 데이터에 캘린더 조인 (영업일 비율 보정)
cal['year'] = cal['date'].dt.year
cal['month'] = cal['date'].dt.month

# 월별 영업일/비영업일 일수
cal_monthly = cal.groupby(['year', 'month']).agg(
    working_days=('is_non_working', lambda x: (x == 0).sum()),
    non_working_days=('is_non_working', 'sum'),
    total_days=('date', 'count')
).reset_index()

yearly_cal = yearly.merge(cal_monthly, on=['year', 'month'], how='left')

# 영업일 보정 실차율 (영업일당 수치)
yearly_cal['trips_per_workday'] = yearly_cal['trip_count'] / yearly_cal['working_days']
yearly_cal['rev_per_workday'] = yearly_cal['pay_sum'] / yearly_cal['working_days']

# 연도별 영업일 보정 집계
yearly_cal_agg = yearly_cal.groupby('year').agg(
    occ_rate=('occ_rate_dist', 'mean'),
    trips_per_wd=('trips_per_workday', 'mean'),
    rev_per_wd=('rev_per_workday', 'mean')
).reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(yearly_cal_agg['year'], yearly_cal_agg['trips_per_wd'], color='#42a5f5', edgecolor='white')
ax.axvline(2018, color='red', ls='--', alpha=0.7, label='카카오T 본격화')
ax.set_title('연도별 영업일당 평균 운행 건수', fontsize=13)
ax.set_xlabel('연도'); ax.set_ylabel('영업일당 운행 건수')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 8. 종합 분석 및 결론

### 분석 결과 요약

#### 실차율 변화
- 카카오T 본격화(2018) 전후 실차율 추이를 거리 기준으로 측정
- Cramer & Krueger (2016)의 미국 Uber 연구에서 보고된 30% 향상과 비교
- 코로나 기간(2020-2022)의 외부 충격 효과를 분리하여 순수 플랫폼 효과 해석 필요

#### 운전자 매출 불평등
- Gini 계수, CV, P90/P10 비율 세 지표로 다각도 측정
- 플랫폼이 수요 매칭을 최적화하면 하위 운전자 매출이 상승하여 불평등이 감소할 수 있음
- 반대로 플랫폼 알고리즘이 고평점 운전자를 우선 배차하면 불평등이 심화될 수도 있음

#### 시간대별 빈차 패턴
- 심야/새벽 빈차율 변화는 플랫폼의 수요-공급 매칭 효율을 직접 반영
- 출퇴근 시간대 vs 비첨두 시간대 효율 차이의 연도별 변화

#### 한계점
- 플랫폼 호출 여부 컬럼이 없어 직접적인 플랫폼 효과 분리 불가
- 코로나 효과와 플랫폼 효과의 완전한 분리 어려움
- 운전자 고유 특성(경력, 근무시간)을 통제하지 못함

### 실무 활용
- **배차 알고리즘 평가**: 실차율과 불평등 지표를 플랫폼 알고리즘의 성과 지표(KPI)로 활용하여, 알고리즘 업데이트의 효과를 정량적으로 모니터링할 수 있다.
- **운전자 소득 보호 정책**: Gini 계수가 상승 추세라면, 최저 운행 보장제 또는 비첨두 시간대 인센티브를 통해 하위 운전자의 매출을 보호하는 정책을 검토할 수 있다.
- **시간대별 공급 최적화**: 빈차율이 높은 시간대를 식별하여 운전자에게 해당 시간대 운행을 권장하거나, 반대로 수요가 낮은 시간대의 공급을 줄이는 탄력적 운영 전략을 수립할 수 있다.

## References

1. Cramer, J., & Krueger, A. B. (2016). Disruptive Change in the Taxi Business: The Case of Uber. *American Economic Review*, 106(5), 177-182. (NBER Working Paper No. 22083)
2. Gini, C. (1912). *Variabilita e mutabilita*. Reprinted in Pizetti, E. & Salvemini, T. (1955), *Memorie di metodologica statistica*. Rome: Libreria Eredi Virgilio Veschi.
3. Hall, J. V., & Krueger, A. B. (2018). An Analysis of the Labor Market for Uber's Driver-Partners in the United States. *ILR Review*, 71(3), 705-732.
4. Castillo, J. C. (2023). Who Benefits from Surge Pricing? *Management Science*, 69(10), 6037-6049.

In [ ]:
mem_usage('final')
print('=== 22_platform_efficiency 분석 완료 ===')